# 13 CNN 图像分类训练流程

前面已经把 CNN 的主要零件学了一遍：

```text
图像基础
-> 卷积层
-> padding 和 stride
-> 多通道卷积
-> 多卷积核
-> 特征图
-> 池化层
-> Flatten
-> 全连接分类头
-> 反向传播
```

现在要把这些知识连成一条完整的训练流程。

这一节暂时不写代码，先理解：

```text
一张图片是怎么进入 CNN 的？
CNN 怎么得到预测结果？
loss 是在哪里产生的？
反向传播和优化器分别做什么？
训练时到底在更新什么？
```

理解这条线之后，再去看图像分类代码，就不会觉得每一行都很突然。

## 1. 为什么要学训练流程

前面学习卷积层、池化层、全连接层时，更多是在看网络里面的局部结构。

但是一个模型真正训练起来时，不是只跑一个卷积层，而是一个完整过程：

```text
准备数据
-> 前向传播
-> 计算损失
-> 反向传播
-> 更新参数
-> 重复很多轮
```

这条流程和之前学 MLP、MNIST 时的训练流程本质是一样的。

区别在于：CNN 更适合处理图像，所以前面多了卷积层、池化层这些专门提取图像特征的结构。

## 2. 图像分类任务在做什么

图像分类任务的目标很简单：

```text
输入一张图片，输出这张图片属于哪个类别。
```

比如手写数字识别：

```text
输入：一张手写数字图片
输出：0 到 9 中的一个类别
```

再比如猫狗分类：

```text
输入：一张动物图片
输出：猫 或 狗
```

所以 CNN 图像分类可以先理解成：

```text
CNN 负责看图
全连接分类头负责根据特征做分类
loss 负责告诉模型错得多不多
优化器负责修改参数
```

## 3. 一张图片进入 CNN 前是什么形状

图片进入 CNN 前，通常会被整理成张量。

如果是一张灰度图，比如 MNIST，可以理解成：

```text
1 x 28 x 28
```

含义是：

```text
1 个通道
高度 28
宽度 28
```

如果是一张 RGB 彩色图，可以理解成：

```text
3 x H x W
```

含义是：

```text
3 个通道：R、G、B
高度 H
宽度 W
```

真正训练时，一般不是一张一张送进去，而是一批一批送进去。

所以常见形状是：

$$
B \times C \times H \times W
$$

其中：

- $B$ 表示 batch size，也就是这一批有多少张图片。
- $C$ 表示通道数。
- $H$ 表示高度。
- $W$ 表示宽度。

## 4. 第一步：数据和标签一起进入训练流程

训练时，每一张图片通常都有一个正确答案，也叫标签。

比如 MNIST：

```text
图片：某张手写数字图片
标签：7
```

训练时模型拿到的是一批数据：

```text
图片 batch
标签 batch
```

图片 batch 用来送进 CNN。

标签 batch 用来和模型预测结果做比较。

所以训练不是让模型随便看图，而是告诉它：

```text
你先预测一下。
然后我拿你的预测和标准答案对比。
错了就根据错误方向调整参数。
```

## 5. 第二步：前向传播得到类别分数

图片进入 CNN 后，会经过一层一层处理：

```text
图片
-> 卷积层：提取局部特征
-> 激活函数：增加非线性表达能力
-> 池化层：压缩特征图
-> 更多卷积和池化
-> Flatten：展平成一维向量
-> 全连接分类头：输出类别分数
```

最后输出的通常不是直接的类别名字，而是一组分数。

比如 10 分类任务，模型可能输出 10 个分数：

```text
类别 0 的分数
类别 1 的分数
类别 2 的分数
...
类别 9 的分数
```

哪个类别分数最高，模型就更倾向于认为图片属于哪个类别。

## 6. 第三步：用 loss 衡量预测错得多不多

模型输出类别分数之后，还不能直接更新参数。

我们需要先把预测结果和真实标签进行比较，得到一个 loss。

loss 可以先理解成：

```text
模型这一次预测的错误程度。
```

如果预测很接近真实标签，loss 通常比较小。

如果预测和真实标签差得很远，loss 通常比较大。

训练 CNN 的目标就是：

$$
\text{让 loss 尽可能变小}
$$

对于分类任务，常见的损失函数是交叉熵损失。

入门阶段先记住它的作用：

```text
交叉熵损失用来衡量分类预测和真实类别之间的差距。
```

## 7. 第四步：反向传播计算梯度

有了 loss 之后，反向传播开始工作。

反向传播要回答的是：

```text
每个可学习参数应该往哪个方向改，才能让 loss 变小？
```

在 CNN 中，需要计算梯度的主要参数包括：

```text
卷积核权重
卷积层 bias
全连接层权重
全连接层 bias
```

反向传播会从 loss 开始，沿着网络反着走：

```text
loss
-> 全连接层
-> Flatten
-> 池化层
-> 激活函数
-> 卷积层
```

它不是为了修改中间的特征图，而是为了计算每个可学习参数的梯度。

## 8. 第五步：优化器根据梯度更新参数

反向传播负责计算梯度。

优化器负责真正修改参数。

如果用最简单的梯度下降思想，可以写成：

$$
\text{新参数} = \text{旧参数} - \text{学习率} \times \text{梯度}
$$

这里的学习率可以理解成：

```text
每次改参数时，步子迈多大。
```

学习率太大，可能一步迈过头。

学习率太小，训练可能很慢。

所以在训练流程中：

```text
loss 告诉模型错得多不多
反向传播算出每个参数怎么影响 loss
优化器按照梯度修改参数
```

## 9. 一个 batch 训练时发生了什么

训练不是只做一次前向传播就结束。

通常一个 batch 的训练可以理解成：

```text
1. 取出一批图片和标签
2. 图片进入 CNN，得到类别分数
3. 类别分数和真实标签计算 loss
4. 根据 loss 进行反向传播，计算梯度
5. 优化器根据梯度更新参数
6. 清理本轮梯度，准备下一批数据
```

其中最关键的关系是：

```text
前向传播：从图片到预测
反向传播：从 loss 回到参数
优化器：真正改参数
```

## 10. 一个 epoch 是什么

一个 batch 只是一小批数据。

如果训练集里有很多图片，模型需要分很多批看完。

当模型把整个训练集完整看过一遍，就叫一个 epoch。

可以这样理解：

```text
batch：一次看一小批
iteration：完成一次参数更新
epoch：完整看完一遍训练集
```

比如训练集有 60000 张图片，batch size 是 100。

那么一个 epoch 大约会有：

$$
60000 \div 100 = 600
$$

次参数更新。

## 11. 训练集和测试集分别有什么用

训练模型时，通常会把数据分成训练集和测试集。

训练集用来更新参数：

```text
模型在训练集上前向传播、计算 loss、反向传播、更新参数。
```

测试集用来检查模型学得怎么样：

```text
模型在测试集上只做预测，不根据测试集修改参数。
```

这个区别很重要。

如果模型只在训练集上表现好，但在测试集上表现差，说明它可能只是把训练数据记住了，没有真正学会一般规律。

这就是之前学过的过拟合问题。

## 12. 训练模式和评估模式

训练时，模型需要更新参数，所以流程包括：

```text
前向传播
计算 loss
反向传播
优化器更新参数
```

评估时，模型只是用来预测，所以流程通常是：

```text
前向传播
计算准确率或 loss
不反向传播
不更新参数
```

如果网络中有 Dropout、BatchNorm 这类层，训练模式和评估模式还会影响它们的行为。

入门阶段先记住：

```text
训练模式：模型在学习，参数会被更新。
评估模式：模型在考试，参数不更新。
```

## 13. 准确率怎么看

分类任务除了看 loss，还经常看准确率。

准确率可以理解成：

```text
预测正确的图片数量 / 总图片数量
```

比如 100 张测试图片中，模型预测对了 92 张，那么准确率就是：

$$
\frac{92}{100} = 92\%
$$

loss 和准确率关注的角度不同：

```text
loss：更细地衡量预测和真实标签之间的差距
准确率：直接看最终类别有没有预测对
```

训练时常见情况是：

```text
loss 逐渐下降
准确率逐渐上升
```

这通常说明模型正在学到有用规律。

## 14. CNN 训练时到底学到了什么

CNN 训练时，真正被不断调整的是参数。

在卷积层中，模型学习的是：

```text
卷积核里的权重和 bias
```

这些卷积核会逐渐变得擅长提取某些图像特征。

比如浅层卷积核可能更容易学到：

```text
边缘
角点
简单纹理
明暗变化
```

深层卷积核可能组合出更复杂的特征：

```text
局部形状
物体部件
更抽象的类别相关特征
```

所以 CNN 不是人为规定每个卷积核必须识别什么。

它是在训练过程中，通过 loss 和反向传播慢慢调整出来的。

## 15. 把训练流程串起来

现在可以把 CNN 图像分类训练过程合成一条完整路线：

```text
图片和标签
-> 图片进入 CNN
-> 卷积层提取特征
-> 池化层压缩特征
-> Flatten 展平
-> 全连接分类头输出类别分数
-> loss 比较预测和真实标签
-> 反向传播计算梯度
-> 优化器更新卷积层和全连接层参数
-> 重复很多 batch 和 epoch
```

这就是 CNN 训练的主线。

后面真正写图像分类代码时，代码基本也是围绕这条线展开。

## 16. 和前面 MLP 训练流程的关系

CNN 和 MLP 的训练流程本质一样：

```text
前向传播
-> 计算 loss
-> 反向传播
-> 优化器更新参数
```

区别主要在模型结构上。

MLP 直接把输入展平成一维向量，再用全连接层处理。

CNN 先保留图像的空间结构，用卷积层提取局部特征，再交给全连接层分类。

所以可以这样记：

```text
MLP：更像直接看一串数字。
CNN：更像先看图像局部结构，再做分类判断。
```

## 17. 本节小结

这一节要记住四句话：

```text
1. CNN 图像分类训练的目标，是让模型根据图片预测正确类别。
2. 前向传播负责从图片得到类别分数和 loss。
3. 反向传播负责计算可学习参数的梯度。
4. 优化器负责更新卷积层和全连接层中的权重、bias。
```

更简单地说：

```text
前向传播看结果。
loss 衡量错误。
反向传播算责任。
优化器改参数。
```

## 18. 自测问题

1. CNN 图像分类任务的输入和输出分别是什么？
2. 为什么训练时图片通常是一批一批送入模型？
3. batch、iteration、epoch 分别是什么意思？
4. loss 在训练流程中起什么作用？
5. 反向传播负责更新参数吗？如果不是，它负责什么？
6. 优化器真正修改的是哪些东西？
7. 训练集和测试集的作用有什么区别？
8. 为什么说 CNN 和 MLP 的训练流程本质一样，但结构不同？